# Classic PROJ grid-registration validation (HEAVY tier)

Proves the **heavy-tier grid-dir propagation to real classic cluster executors** end to end.

The source CRS references a synthetic NTv2 grid by filename (`+nadgrids=synthetic.gsb`), which
applies a known **+30 arc-second latitude** shift. PROJ can only build the transform if the
heavy `gbx_st_transformcrs` JVM expression on the *executor* locates `synthetic.gsb` on its
search path. Task-11 (`ExpressionConfigExpr` + `GDALManager.init`) embeds the grid dir in
the registry and injects it on Task-11 `GDALManager.init(extra_proj_dirs=...)` at executor task time.

We fan a point across N partitions with `repartition(N, "id")` so multiple executor tasks
each independently build a transformer and must each find the grid. Getting the exact shift
on **every** partition is the proof the grid dir reached every executor.

## Assertions
- **Assertion 1 (VECTOR path, Task-11 gap proof):** In a session that has used NO raster fn
  (so `GDALManager.init` would not otherwise have run), run `gbx_st_transformcrs` with the grid CRS.
  Assert lat == 51.5083333 on EVERY partition. This exercises Task-11 `GDALManager.init`
  injection into `ST_TransformCrs`.
- **Assertion 2 (RASTER path):** Run `gbx_rst_transformcrs` (gdalwarp/`SetConfigOption` path)
  against the same grid CRS on a sample raster.


In [ ]:
import json

GRID_DIR = "/Volumes/geospatial_docs/geobrix/sample-data/proj-grids"
GRID_FILE = GRID_DIR + "/synthetic.gsb"

SRC_CRS = "+proj=longlat +ellps=GRS80 +nadgrids=synthetic.gsb +no_defs"
TGT_CRS = "EPSG:4326"
PT_WKT = "POINT (0 51.5)"
SHIFT_SEC = 30.0
EXPECT_LON = 0.0
EXPECT_LAT = 51.5 + SHIFT_SEC / 3600.0  # 51.508333...
N_PART = 3

results = {
    "tier": "heavy",
    "grid_file": GRID_FILE,
    "expect_lat": EXPECT_LAT,
    "stages": {},
    "vector_result": None,
    "raster_result": None
}

import os

assert os.path.isfile(GRID_FILE), f"grid fixture not staged/visible on driver: {GRID_FILE}"
print("driver sees grid fixture:", GRID_FILE, os.path.getsize(GRID_FILE), "bytes")

In [ ]:
# Register the HEAVY tier (vectorx + rasterx JVM expressions).
# Then register the grid dir, which will re-register gbx_st_transformcrs
# with GRID_DIR embedded in the JVM expression's ExpressionConfigExpr.

# Import heavy-tier registration
from databricks.labs.gbx.vectorx import functions as vectorx_functions
from databricks.labs.gbx.rasterx import functions as rasterx_functions
from databricks.labs.gbx import crs_grids
from databricks.labs.gbx.core import proj_grids

# Clean slate
proj_grids.set_registered_dirs([], replace=True)

# Register both heavy tiers so ST_TransformCrs and RST_TransformCrs are JVM expressions
vectorx_functions.register(spark)
rasterx_functions.register(spark)

# Now register the grid directory, which will re-register the expressions
# with the grid dir embedded in ExpressionConfigExpr
registered = crs_grids.register_proj_grids(spark, GRID_DIR)
print("registered dirs:", registered)
results["registered"] = registered
assert GRID_DIR in registered, f"expected {GRID_DIR} in registered dirs, got {registered}"

print("✓ Heavy tier registered with grid dir")

In [ ]:
# ASSERTION 1: VECTOR path with Task-11 GDALManager.init injection.
# Fan a point across N partitions and transform with the grid-dependent CRS.
# Extract lon/lat via the PRODUCT ST built-ins (ST_GeomFromWKB / ST_X / ST_Y) so the
# notebook needs NO extra Python deps (no shapely) — keeps the run env-robust. The
# heavy gbx_st_transformcrs expression is still the thing under test.

from pyspark.sql import functions as F

base = (
    spark.range(N_PART)
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("geom", F.lit(PT_WKT))
    .repartition(N_PART, "id")  # force distribution across executor tasks
    .selectExpr(
        "id",
        "spark_partition_id() AS pid",
        f"gbx_st_transformcrs(geom, '{TGT_CRS}', '{SRC_CRS}') AS out_wkb",
    )
)
df = base.selectExpr(
    "id", "pid", "out_wkb",
    "CASE WHEN out_wkb IS NULL THEN NULL ELSE ST_X(ST_GeomFromWKB(out_wkb)) END AS lon",
    "CASE WHEN out_wkb IS NULL THEN NULL ELSE ST_Y(ST_GeomFromWKB(out_wkb)) END AS lat",
)
rows = df.collect()
n_part_seen = len(set(r["pid"] for r in rows))
print(f"collected {len(rows)} rows across {n_part_seen} partitions")

# Do NOT assert-abort on NULL — record it, so the notebook ALWAYS reaches the JSON exit
# (a NULL is exactly the failure mode under test: grid not found on that executor task).
lats = []
nulls = []
for r in rows:
    if r["out_wkb"] is None or r["lat"] is None:
        nulls.append(r["pid"])
        continue
    lats.append((r["pid"], float(r["lon"]), float(r["lat"])))

results["stages"]["vector_treated"] = [{"pid": p, "lon": x, "lat": y} for (p, x, y) in lats]
for p, x, y in lats:
    print(f"  pid={p}  lon={x:.8f}  lat={y:.8f}")
if nulls:
    print(f"  NULL (grid not found) on partitions: {nulls}")

TOL = 1e-6
bad_vector = [
    (p, x, y) for (p, x, y) in lats
    if abs(x - EXPECT_LON) > TOL or abs(y - EXPECT_LAT) > TOL
]
vector_pass = (not bad_vector) and (not nulls) and (n_part_seen >= 1)
results["vector_result"] = {
    "n_partitions": n_part_seen,
    "n_rows": len(rows),
    "null_partitions": nulls,
    "bad": bad_vector,
    "pass": vector_pass,
}

print("\n=== VECTOR RESULT ===")
print(json.dumps(results["vector_result"], indent=2))
if not vector_pass:
    results["error"] = f"VECTOR: {len(bad_vector)} bad + {len(nulls)} NULL partition(s) (expected lat {EXPECT_LAT})"
    print(f"\nFAIL: {results['error']}")
else:
    print(f"\nPASS: registered PROJ grid consulted on all {n_part_seen} executor partitions.")

In [ ]:
# ASSERTION 2: RASTER path (gbx_rst_transformcrs with grid CRS)
# Use a sample raster from the Volume and transform it with the grid-dependent CRS.
# This exercises the heavy raster path (GDAL Warp + SetConfigOption).

# Load a sample raster from the Volume
SAMPLE_RASTER = "/Volumes/geospatial_docs/geobrix/sample-data/london/sentinel2/london_sentinel2_red.tif"

try:
    # Check if the raster exists
    import os as os_raster
    if not os_raster.path.exists(SAMPLE_RASTER):
        # Try the light validation raster path instead
        SAMPLE_RASTER = "/Volumes/main/geobrix_samples/geobrix-examples/london/raster/london_sentinel2_red.tif"
    
    if not os_raster.path.exists(SAMPLE_RASTER):
        raise FileNotFoundError(f"Sample raster not found at {SAMPLE_RASTER}")
    
    print(f"Using raster: {SAMPLE_RASTER}")
    
    # Read raster into a DataFrame
    raster_df = spark.read.format("gdal") \
        .option("driver", "gtiff_gdal") \
        .load(SAMPLE_RASTER)
    
    print(f"Raster schema: {raster_df.schema}")
    print(f"Raster count: {raster_df.count()} tile(s)")
    
    # Simple transform test: just verify the function runs without error
    # (full raster comparison is complex and out of scope for this proof)
    result_df = raster_df.selectExpr(
        "rst",
        f"gbx_rst_transformcrs(rst, '{TGT_CRS}', '{SRC_CRS}') AS transformed"
    )
    
    transformed_count = result_df.count()
    print(f"Transform succeeded: {transformed_count} tile(s) produced")
    
    results["raster_result"] = {
        "n_tiles": transformed_count,
        "pass": transformed_count > 0,
        "note": "raster transform completed (grid CRS applied)"
    }
    
except FileNotFoundError as e:
    print(f"Raster assertion skipped: {e}")
    results["raster_result"] = {
        "skipped": True,
        "reason": str(e)
    }
except Exception as e:
    print(f"Raster assertion failed: {e}")
    import traceback
    traceback.print_exc()
    results["raster_result"] = {
        "pass": False,
        "error": str(e)
    }

print("\n=== RASTER RESULT ===")
print(json.dumps(results["raster_result"], indent=2))

In [ ]:
# Final summary — ALWAYS emit structured JSON via dbutils.notebook.exit.
# (Never raise before the exit: a raise loses the JSON and leaves only a traceback,
#  which is exactly what masked the first run.)
print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

vector_pass = bool((results.get("vector_result") or {}).get("pass", False))
_rr = results.get("raster_result") or {}
raster_pass = bool(_rr.get("pass", False))
raster_skipped = bool(_rr.get("skipped", False))

# The Task-11 heavy VECTOR proof is the definitive result of this validation;
# the raster path is a secondary check and does not gate overall_pass (so a raster
# reader hiccup can never mask the vector proof). Its status is reported separately.
results["overall_pass"] = vector_pass
results["summary"] = (
    f"VECTOR: {'PASS' if vector_pass else 'FAIL'}, "
    f"RASTER: {'PASS' if raster_pass else 'SKIP' if raster_skipped else 'FAIL'}"
)

print(json.dumps(results, indent=2))
print(("\nPASS" if vector_pass else "\nFAIL") + f": {results['summary']}")

dbutils.notebook.exit(json.dumps(results))